In [ ]:
import plotly.graph_objects as pg
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Displacement Peak History Generation

def dispProfile(test_data):
    dispPeakHist = test_data[:,0]
    # Displacement Increment History Generation    

    incr = 0.1
    dispIncrHist = np.array([])
    for i in range(0,dispPeakHist.shape[0]-1):
        if (dispPeakHist[i+1] > dispPeakHist[i]):
            incrModified = incr
        else:
            incrModified = -incr

        IncrHist = np.arange(dispPeakHist[i], dispPeakHist[i+1], incrModified)
        dispIncrHist = np.append(dispIncrHist, IncrHist) 
        
    return dispIncrHist

test_data = pd.read_excel('test_dataa.xlsx')
test_data = test_data.to_numpy()

figure_roll = pg.Figure()
figure_roll.add_trace(pg.Scatter(x=np.arange(0,dispProfile(test_data).shape[0],1),
                                    y=dispProfile(test_data)))
figure_roll.show()

plt.plot(np.arange(0,dispProfile(test_data).shape[0],1), dispProfile(test_data))
plt.xlabel('Steps')
plt.ylabel('Displacement (mm)')


import openseespy.opensees as ops
import opsvis as opsv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as pg

# --- INICIALIZAÇÃO DO MODELO ---
ops.wipe()
ops.model('basic', '-ndm', 2, '-ndf', 3)

m = 1000 # 1m = 1000mm
kN = 1000 # 1kN =  1000N

nDim = 2
nDof = 3
nEle = 10 #pode diminuir 
# Geometria do pilar
B = 0.55*m #mm
H = 0.55*m #m
L = 1.65*m #m
A = B*H #mm2
E = 2e5 #N/mm2
Iz = B*(H**3)/12
# cobrimento = 0.04*m # mm
#rho_concreto = 2500 # kg/m3

P_axial = -968*kN #N
P_lateral = 1 

nNodes = nEle + 1
nCoords = np.zeros([nNodes, nDim])

eleTags = []
nodeTags = []

for i in range (0, nNodes):
    nCoords[i,0] = 0 # x
    nCoords[i,1] = i*(L/nEle) # y
    nodeTags.append(i+1)

for i in range (0,nEle):
    eleTags.append(i+1)

for i in range (0, nNodes):
    ops.node(nodeTags[i], nCoords[i,0], nCoords[i,1])

# Aço Longitudinal
matTag_aco = 1
Fy = 511 #MPa resistência ao escoamento
E0 = 200e3 #MPa mod de elasticidade
b = 0.01
diam_longitudinal = 0.020*m # mm
barras_camadas = [5, 2, 2, 5] 
area_s = np.pi * (diam_longitudinal**2) / 4 
Ast = sum(barras_camadas) * area_s 
ops.uniaxialMaterial('Steel01', matTag_aco, Fy, E0, b)

# Aço Transversal 
diam_transversal = 0.012*m # mm
fy_transv = 325 # MPa)
npernas_yy = npernas_xx = 4



#0.6228*np.sqrt(fc)*(10**6)

# --- Concreto Não Confinado (Concrete02) ---
matTag_NC = 2
fc_inicial = 32 #MPa   
epsc0_nc = 0.00208
fpcu_nc = 0.2 * fc_inicial 
epsU_nc = 0.008
# 
rat = 0.6  # Fator de redução de rigidez de recarregamento
#ft = 3.0e6 # Resistência à tração (exemplo: 3 MPa)
ft = 0.6228*np.sqrt(fc_inicial)
Ets = ft / 0.002 # Módulo de Elasticidade após a tração

ops.uniaxialMaterial('Concrete02', matTag_NC, -fc_inicial, -epsc0_nc, -fpcu_nc, -epsU_nc, rat, ft, Ets) 


print(ft)

# --- Concreto Confinado 1 (S1 = 0.11m) - Região Crítica ---
#espacamento_s_1 = 0.11 # m
#fc_c1, ec_c1, epsu_c1 = calc_confined_concrete(fc_inicial, epsc0_nc, cobrimento, diam_transversal, diam_longitudinal, npernas_xx, npernas_yy, espacamento_s_1, B, H, fy_transv, Ast)
matTag_C1 = 3
fc_c1 = 40.45
ec_c1 = 0.0048
fpc_u_C1 = fc_c1*0.2
epsc_u_C1 = 0.008
ops.uniaxialMaterial('Concrete02', matTag_C1, -fc_c1, -ec_c1, -fpc_u_C1, -epsc_u_C1, rat, ft, Ets)

# --- Concreto Confinado 2 (S2 = 0.22m) - Região Meio/Topo ---
#espacamento_s_2 = 0.22 # m
#fc_c2, ec_c2, epsu_c2 = calc_confined_concrete(fc_inicial, epsc0_nc, cobrimento, diam_transversal, diam_longitudinal, npernas_xx, npernas_yy, espacamento_s_2, B, H, fy_transv, Ast)
matTag_C2 = 4
fc_c2 = 35.3
ec_c2 = 0.0032
fpc_u_C2 = fc_c2*0.2
epsc_u_C2 = 0.008
ops.uniaxialMaterial('Concrete02', matTag_C2, -fc_c2, -ec_c2, -fpc_u_C2, -epsc_u_C2, rat, ft, Ets)

#SEÇÃO TRANSVERSAL

c = 0.04*m # cover (mm)
# area_s = 0.0006446 
d = 0.02*m #20mm de diam
area_s = np.pi*(d**2)/4 #barra de aço - longitudinal (m)

secTag = 1
fib_sec_1 = [['section', 'Fiber', 1, '-GJ', 1.0e6],
             ['patch', 'quad', matTag_C1, 20, 1, (-H/2)+c, (-B/2)+c, (H/2)-c, (-B/2)+c, (H/2)-c, (B/2)-c, (-H/2)+c, (B/2)-c], #confinado - 5
             ['patch', 'quad', matTag_NC, 2, 1, (-H/2), (-B/2), (-H/2)+c, (-B/2), (-H/2)+c, (B/2), (-H/2), (B/2)], #1 não confinado
             ['patch', 'quad', matTag_NC, 6, 1, (-H/2)+c, (B/2)-c, (H/2)-c, (B/2)-c, (H/2)-c, (B/2), (-H/2)+c, (B/2)], #2
             ['patch', 'quad', matTag_NC, 6, 1, (-H/2)+c, (-B/2), (H/2)-c, (-B/2), (H/2)-c, (-B/2)+c, (-H/2)+c, (-B/2)+c], #3
             ['patch', 'quad', matTag_NC, 2, 1, (H/2)-c, (-B/2), (H/2), (-B/2), (H/2), (B/2), (H/2)-c, (B/2)], #4
             ['layer', 'straight', matTag_aco, 4, area_s, (H/2)-c, (B/2)-c, (H/2)-c, (-B/2)+c],
             ['layer', 'straight', matTag_aco, 2, area_s, (H-2*c)/6, (B/2)-c, (H-2*c)/6, (-B/2)+c],
             ['layer', 'straight', matTag_aco, 2, area_s, -(H-2*c)/6, (B/2)-c, -(H-2*c)/6, (-B/2)+c],
             ['layer', 'straight', matTag_aco, 4, area_s, (-H/2)+c, (B/2)-c, (-H/2)+c, (-B/2)+c]]
opsv.fib_sec_list_to_cmds(fib_sec_1)
matcolor = ['r', 'lightgrey', 'gold', 'w', 'w', 'w']
opsv.plot_fiber_section(fib_sec_1, matcolor=matcolor)
plt.axis('equal')
plt.savefig('fibsec_rc.png')
plt.show()

transfTag = 1
ops.geomTransf('Linear', transfTag)

integrationTag = 1
nIntPts = 5
ops.beamIntegration('Lobatto', integrationTag, secTag, nIntPts)

formulation = 'disp'  # mudar para 'disp' conforme necessário OBS continuar no disp 
#não precisa de 5 pontos de integração

for i in range(nEle):
    nodes = [nodeTags[i], nodeTags[i+1]]
    eleTag = eleTags[i]

    if formulation.lower() == 'force':
        ops.element('forceBeamColumn', eleTag, *nodes, transfTag, integrationTag, '-iter', 100, 1e-8)
        
    else:
        ops.element('dispBeamColumn', eleTag, *nodes, transfTag, integrationTag, '-cMass', '-mass', 0.0)

ops.fix(nodeTags[0], 1, 1, 1)

ops.recorder('Node', '-file', 'ReacDOF1_c1.out', '-node', nodeTags[0],  '-dof', 1, 'reaction')  
ops.recorder('Node', '-file', 'DispDOF1_c1.out', '-node', nodeTags[-1], '-dof', 1, 'disp')    

opsv.plot_model()
plt.xlabel('Largura')
plt.ylabel('Altura')
plt.title('Modelo Indeformado')
plt.show()

#CARGA AXIAL

timeSeriesTag = 1
patternTags = [1,2]
ops.timeSeries('Linear', timeSeriesTag)
ops.pattern('Plain', patternTags[0], timeSeriesTag)
ops.load(nodeTags[-1], 0, P_axial, 0)

ops.constraints('Plain')
ops.numberer('RCM')
ops.system('BandGeneral')
ops.test('NormDispIncr', 1e-5, 10)
ops.algorithm('Newton')

nSteps = 10
dt = 1/nSteps
ops.integrator('LoadControl', dt)
ops.analysis('Static')
ok = ops.analyze(nSteps)
if (ok == 0):
    print('Carga axial ok')
else:
    print('Carga axial falhou')

ops.loadConst('-time', 0.0)

ops.pattern("Plain", patternTags[1], timeSeriesTag)    
ops.load(nodeTags[-1], P_lateral, 0.0, 0.0)  

ops.wipeAnalysis()           
ops.constraints("Plain")     
ops.numberer("RCM")          
ops.system('BandGeneral')     
ops.test('NormDispIncr', 1.0e-5, 10)    
                                         
ops.algorithm('Newton')   


test_data = pd.read_excel('test_dataa.xlsx')
test_data = test_data.to_numpy()
dispIncrHist = dispProfile(test_data)
dofLatLoad = 1

d0=0.0
for i in range(1, dispIncrHist.shape[0]):
    d1 = dispIncrHist[i]
    dt = d1-d0
    ops.integrator("DisplacementControl",  nodeTags[-1], dofLatLoad, dt)
    ops.analysis("Static")
    pushoverAnalysisStatus = ops.analyze(1)
    if (pushoverAnalysisStatus < 0):
        break
    else:
        d0 = d1 # next step

    
if (pushoverAnalysisStatus == 0):
    print('Pushover Analysis Successfull')
else:
    print("Pushover Analysis Failed at {} step.".format(i))

freeDisp = np.loadtxt('DispDOF1_c1.out')
baseReaction = np.loadtxt('ReacDOF1_c1.out')/1000

plt.plot(freeDisp[10:], -baseReaction[10:], 'b-', label='Numerical Simulation')
plt.plot(test_data[:,0], test_data[:,1], 'r-', label='Test Data')

plt.xlabel('Displacement (mm)')
plt.ylabel('Force (kN)')
plt.title('Force Displacement Response')
plt.legend()
plt.grid(True)
plt.show()


: 